> **AI-assisted:** Review paths before running.

# Data schema audit (Drive decompress + schema + doc check)

**Run this notebook in Google Colab** (Runtime → Run all). Colab only executes Python; all files stay on Drive.

## Layout on Drive

```text
MyDrive/CSCC01 - Group Project/Data/
├── CRMP-full-data.zip
├── CRMP-full-data/                    ← outer zip extracts here
├── decompressor/
│   └── audit_data_schema.py           ← engine (you placed it here)
├── data_schema_audit_report.txt       ← report output
└── (optional) audit_data_schema.ipynb ← or open ipynb from GitHub in Colab
```

**Important:** After updating the repo, re-upload `audit_data_schema.py` to `Data/decompressor/` on Drive (Colab reads the Drive copy, not GitHub).

In [ ]:
# Cell 1: Mount Drive
try:
    from google.colab import drive  # pyright: ignore[reportMissingImports]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    print("[OK] Drive mounted")
else:
    print("[WARN] Local run — set paths in Cell 3")

In [ ]:
# Cell 2: Load audit_data_schema.py from Drive (runpy — reliable in Colab)
# IMPORTANT: Re-upload the latest scripts/audit_data_schema.py from the repo to Drive
# after each update — Colab does NOT read GitHub automatically.
import os
import runpy
import types

DRIVE_DATA = "/content/drive/MyDrive/CSCC01 - Group Project/Data"
AUDIT_SCRIPT = f"{DRIVE_DATA}/decompressor/audit_data_schema.py"

if not os.path.isfile(AUDIT_SCRIPT):
    raise FileNotFoundError(
        f"Expected script at:\n  {AUDIT_SCRIPT}\n"
        "Upload the latest audit_data_schema.py to Data/decompressor/ on Drive."
    )

_audit_ns = runpy.run_path(AUDIT_SCRIPT, run_name="audit_data_schema")
audit = types.SimpleNamespace(
    **{k: v for k, v in _audit_ns.items() if not k.startswith("_")}
)
print(f"[OK] Loaded {AUDIT_SCRIPT}")
print(f"[OK] run_audit ready: {callable(getattr(audit, 'run_audit', None))}")

In [ ]:
# Cell 3: Configuration
from pathlib import Path

DRIVE_DATA = Path("/content/drive/MyDrive/CSCC01 - Group Project/Data")

ZIP_FILENAME = "CRMP-full-data.zip"
EXTRACT_ON_DRIVE = IN_COLAB
# Set False on Colab if nested zips are already extracted (saves many minutes on Drive I/O)
EXTRACT_NESTED_ZIPS = False

ROOT_CANDIDATES = [
    DRIVE_DATA / ZIP_FILENAME,
    DRIVE_DATA / "CRMP-full-data",
    DRIVE_DATA / "CRMP-full-data" / "CRMP-full-data",
]

OUTPUT_PATH = DRIVE_DATA / "data_schema_audit_report.txt"
WORK_DIR = DRIVE_DATA if EXTRACT_ON_DRIVE else DRIVE_DATA


def resolve_root(candidates):
    for path in candidates:
        if path.exists():
            print(f"[OK] Root: {path}")
            return path
    raise FileNotFoundError(
        f"Nothing found under {DRIVE_DATA}. Add shortcut to My Drive and upload {ZIP_FILENAME}."
    )


ROOT_PATH = resolve_root(ROOT_CANDIDATES)
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
print(f"[OK] EXTRACT_ON_DRIVE={EXTRACT_ON_DRIVE}; work/extract base: {WORK_DIR}")

In [ ]:
# Cell 4: Run audit (extract nested zips + schema sample + doc checks)
report = audit.run_audit(
    ROOT_PATH,
    work_dir=WORK_DIR,
    extract_on_drive=EXTRACT_ON_DRIVE,
    extract_nested=EXTRACT_NESTED_ZIPS,
    output_path=OUTPUT_PATH,
)

print(report[:5000])
if len(report) > 5000:
    print(f"\n... preview truncated; full report: {OUTPUT_PATH} ({len(report):,} chars)")

In [ ]:
# Cell 5: Optional — download report (already saved on Drive at OUTPUT_PATH)
if IN_COLAB:
    from google.colab import files  # pyright: ignore[reportMissingImports]

    print(f"[INFO] Full report on Drive: {OUTPUT_PATH}")
    print("[INFO] Download to laptop only if needed:")
    # files.download(str(OUTPUT_PATH))
else:
    print(f"[OK] Report at {OUTPUT_PATH}")